In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from numpy.linalg import eig
import pandas as pd

#import matplotlib  
#matplotlib.use('Agg')  # Use a non-GUI backend

In [2]:
cat='C6'
#path=f'C:/shehani/postdoc_work/ML_Diffusion/cal_md/new_cals/{cat}_rel/zbox/'
path=f'C:/shehani/postdoc_work/ML_Diffusion/cal_md/new_cals/{cat}_rel/'
z=20.0
#path='C:/shehani/postdoc_work/ML_Diffusion/md_data/c4_nmeth_nh2o_box1/c2_relaxed_48/'

In [3]:
def oh_xyz(nrep, nsteps):
    #nrep = number of replicas
    #nsteps = number of steps in fs/MD_freq (ex: for 10 ps simulation with MD_freq=10, nsteps=10,000/10=1000)

    #extracting x,y, z cordinates from oh_id.dat files
    x_oh = np.zeros((nrep,nsteps))
    y_oh = np.zeros((nrep,nsteps))
    z_oh = np.zeros((nrep,nsteps))

    for i in range(nrep):
        with open(path+ f'oh_id_{i+1}.dat', 'r') as oh_id:
            xyz= oh_id.readlines()[:nsteps]
            
            for j in range(len(xyz)):                
                x_oh[i,j]=float(xyz[j].split()[2])
                y_oh[i,j]=float(xyz[j].split()[3])
                z_oh[i,j]=float(xyz[j].split()[4])
                
    #dtime=  np.arange(0.01, (0.01*ndt+0.01), 0.01)
    return(x_oh, y_oh, z_oh)




In [4]:
def oh_D(x_oh, y_oh, z_oh, nrep, nsteps, natoms, nsheet, nwater, noh, xbox, ybox, zbox, ndt):
    #nrep = number of replicas
    #nsteps = number of steps in fs/MD_freq (ex: for 10 ps simulation with MD_freq=10, nsteps=10,000/10=1000)
    #natoms = total number of atoms in the sysytem
    #nsheet = total number of atoms in the graphene sheet
    #nwater = total number of water molecules
    #ncat   = total number of atoms in the cation/s
    #noh    =number of OH anions in the system
    #xbox, ybox, zbox = simulation box dimensions through x, y, z
    #ndt = maximum tau value in fs/MDfreq (ex: for Tau = 5000 fs with MD_freq=10, ndt=5000/10=500 )

    #x_oh, y_oh, z_oh= oh_xyz(nrep, nsteps)
    
    MD_freq =10 #prints after every 10 steps (10 fs)
    dt =1.0 #in fs 
    
    nox = nwater + noh
    nhy = 2*nwater + noh
  

    msd_df = pd.DataFrame(columns=['Tau', 'dtime','msd_xx', 'msd_xy', 'msd_xz', 'msd_yy', 'msd_yz', 'msd_zz'])
    msd_df = msd_df.astype(float)
    

    dtime = np.empty(ndt)
    dtime=  np.arange(0.01, (0.01*ndt+0.01), 0.01)

    Dfxx = np.zeros(nrep)  
    Dfyy = np.zeros(nrep)
    Dfzz = np.zeros(nrep)
    D = np.zeros(nrep)
    Df2d = np.zeros(nrep)
    Dfxy = np.zeros(nrep)
    Dfxz = np.zeros(nrep)
    Dfyz = np.zeros(nrep)
    Dfx2d = np.zeros(nrep)
    Dfy2d = np.zeros(nrep)
    Df2d = np.zeros(nrep)
    D = np.zeros(nrep)
    #Dmean = np.zeros(nrep)
    #D_std = np.zeros(nrep)
    #std_err = np.zeros(nrep)
    

    for ll in range(nrep):
        for kk in range(ndt):
            msdxx = 0.0
            msdyy = 0.0
            msdzz = 0.0
            msdxy = 0.0
            msdxz = 0.0
            msdyz = 0.0
            
            for jj in range(math.ceil((nsteps/(kk+1))-1)): ##!!!recheck when kk=499

                msd_df.loc[kk, 'Tau']=kk+1
                msd_df.loc[kk, 'dtime']=(kk+1)/100
                msdxx = (x_oh[ll,(jj*(kk+1)+kk+1)]-x_oh[ll,jj*(kk+1)])**2 +msdxx
                msd_df.loc[kk, 'msd_xx']=msdxx
                msdyy = (y_oh[ll,(jj*(kk+1)+kk+1)]-y_oh[ll,jj*(kk+1)])**2+msdyy
                msd_df.loc[kk, 'msd_yy']=msdyy
                msdzz = (z_oh[ll,(jj*(kk+1)+kk+1)]-z_oh[ll,jj*(kk+1)])**2+msdzz
                msd_df.loc[kk, 'msd_zz']=msdzz
                msdxy = ((x_oh[ll,(jj*(kk+1)+kk+1)]-x_oh[ll,jj*(kk+1)])*(y_oh[ll,(jj*(kk+1)+kk+1)]-y_oh[ll,jj*(kk+1)]))+msdxy
                msd_df.loc[kk, 'msd_xy']=msdxy
                msdxz = ((x_oh[ll,(jj*(kk+1)+kk+1)]-x_oh[ll,jj*(kk+1)])*(z_oh[ll,(jj*(kk+1)+kk+1)]-z_oh[ll,jj*(kk+1)]))+msdxz
                msd_df.loc[kk, 'msd_xz']=msdxz
                msdyz= ((y_oh[ll,(jj*(kk+1)+kk+1)]-y_oh[ll,jj*(kk+1)])*(z_oh[ll,(jj*(kk+1)+kk+1)]-z_oh[ll,jj*(kk+1)]))+msdyz
                msd_df.loc[kk, 'msd_yz']=msdyz

            msd_df.iloc[kk, 2:] = msd_df.iloc[kk, 2:]/(((math.ceil(nsteps/(kk+1))-1)))

        #perform linear fit
        fitxx=np.polyfit(msd_df['dtime'],msd_df['msd_xx'],1)
        fityy=np.polyfit(msd_df['dtime'],msd_df['msd_yy'],1)
        fitzz=np.polyfit(msd_df['dtime'],msd_df['msd_zz'],1)
        fitxy=np.polyfit(msd_df['dtime'],msd_df['msd_xy'],1)
        fitxz=np.polyfit(msd_df['dtime'],msd_df['msd_xz'],1)
        fityz=np.polyfit(msd_df['dtime'],msd_df['msd_yz'],1)



        #generate 3D diffusion matrix
        dxx = fitxx[0]/2
        dyy = fityy[0]/2
        dzz = fitzz[0]/2
        dxy = fitxy[0]/2
        dxz = fitxz[0]/2
        dyz = fityz[0]/2

    
        #generate 2D diffusion matrix
        d_2d = np.array([[dxx, dxy], 
                         [dxy, dyy]])
        w_2d,v_2d=eig(d_2d)


        Dfxx[ll] = dxx
        Dfyy[ll] = dyy
        Dfzz[ll] = dzz
        Dfxy[ll] = dxy
        Dfxz[ll] = dxz
        Dfyz[ll] = dyz
        Dfx2d[ll] = w_2d[0]
        Dfy2d[ll] = w_2d[1]
        Df2d[ll] = ((w_2d[0]+w_2d[1])/2)
        D[ll]= (dxx+dyy)/2
        
    Dmean=np.mean(D)
    D_std= np.std(D)
    std_err= D_std/np.sqrt(nrep)
    

        


    #save msd_df
    msd_df.to_csv(path+"individual_msd_df.csv", index=False)
    return(D, Dmean, D_std, std_err)

        
        
        




In [5]:
nrep=15
steps=2000

x_oh, y_oh, z_oh= oh_xyz(nrep, steps)
d, dmean,dstd, std_err=oh_D(x_oh, y_oh, z_oh, nrep, steps, 458, 287, 46, 1, 15.198, 13.392, 15,500)
            
d, dmean,dstd, std_err

(array([1.71078865, 1.95379124, 0.45894867, 1.00854486, 0.63011862,
        1.5924145 , 1.20977556, 0.79269102, 0.96276335, 0.53533848,
        0.89543811, 0.66405603, 1.07417423, 1.02085844, 0.34493716]),
 0.9903092605325587,
 0.4514585062013858,
 0.11656608506810966)